# IMPORTS

In [ ]:
import os

from PIL import Image
from transformers import AutoModelForObjectDetection, AutoImageProcessor
import onnx
import onnxruntime as ort
import numpy as np
from torch.export import Dim
import torch
from torch import nn

# INPUT DATA PREVIEW

In [ ]:
IMAGE_PATH = "input_image.png"

IMG = Image.open(IMAGE_PATH)
IMG

# LOAD THE PRETRAINED MODEL

In [ ]:
MODEL_CHECKPOINT = "PaddlePaddle/PP-DocLayout_plus-L_safetensors"

model = AutoModelForObjectDetection.from_pretrained(MODEL_CHECKPOINT)
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

# TRAIN/FINE-TUNE (OPTIONAL/ADD LATER)

# TEST MODEL

#### Set the model to evaluation mode

In [ ]:
model.eval() 

#### Prepocess the input

In [ ]:
inputs = processor.preprocess(images = IMG,return_tensors = 'pt')

#### Run the model

In [ ]:
import torch
with torch.no_grad():
    outputs = model(**inputs)

#### Post-process the outputs

In [ ]:
help(processor.post_process_object_detection)

In [ ]:
width, height = IMG.size
results = processor.post_process_object_detection(
    outputs,
    target_sizes = [(height, width)] # rescale the bounding box cordintates as per image size
)

# INSPECT RESULTS

In [ ]:
print(f'{type(results) = }')
print(f'{len(results) = }')
print('Note that result corresponds to number of input batch size')
print()
print(f'{type(results[0]) = }')
print(f'{results[0].keys() = }')
print("This result corresponds to different scores,labels, and boxes corresponding to the detected blocks in the 1st (and only) input image")

In [ ]:
scores = results[0]['scores']
labels = results[0]['labels']
boxes = results[0]['boxes']
print(f'{type(scores) = }, {scores.shape = }')
print(f'{type(labels) = }, {labels.shape = }')
print(f'{type(boxes) = }, {boxes.shape = }')
print("Total 24 blocks are identified with 24 scores and labels. Each block bounding box represented by 4 values in boxes")

#### Plot a segment

In [ ]:
i = 14
num_blocks_detected = len(results[0]['labels'])
i = i % num_blocks_detected

score_i = results[0]['scores'][i].item()
label_i = results[0]['labels'][i].item()
box_i = results[0]['boxes'][i].tolist()
top_left_x, top_left_y,bottom_right_x, bottom_right_y = [round(val,2) for val in box_i]

IMG.crop((top_left_x, top_left_y,bottom_right_x, bottom_right_y ))

# EXPORT THE MODEL to ONNX FORMAT

#### export

In [ ]:
# help(outputs)

In [ ]:
# ONNX_SAVE_PATH = "doclayout.onnx"

# image = Image.open(IMAGE_PATH).convert("RGB")
# pt_inputs = processor(images=image, return_tensors="pt")
# pixel_values = pt_inputs["pixel_values"]

# # set model to the evaluation mode
# model.eval()

# torch.onnx.export(
#     model,
#     (pixel_values,),
#     ONNX_SAVE_PATH,
#     input_names = ["pixel_values"],
#     output_names = ["logits","pred_boxes"],
#     dynamic_shapes={
#         "pixel_values": {0: "batch", 2: "height", 3: "width"},
#     },
#     dynamo = True,
# )



Exporting with dynamic batch option

In [ ]:
os.makedirs("model", exist_ok=True)

ONNX_SAVE_PATH = "./model/doclayout.onnx"


class RTDetrForOnnx(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        return outputs.logits, outputs.pred_boxes


image = Image.open(IMAGE_PATH).convert("RGB")
pt_inputs = processor(images=image, return_tensors="pt")
pixel_values = pt_inputs["pixel_values"]

# Optional: reduces extra intermediate outputs used by auxiliary loss.
model.config.auxiliary_loss = False
model.eval()

export_model = RTDetrForOnnx(model)
export_model.eval()

# Dynamic input dimensions.
batch = Dim("batch", min=1)
height = Dim("height", min=1)
width = Dim("width", min=1)

torch.onnx.export(
    export_model,
    (pixel_values,),
    ONNX_SAVE_PATH,
    input_names=["pixel_values"],
    output_names=["logits", "pred_boxes"],
    dynamo=True,
    dynamic_shapes={
        "pixel_values": {0: batch, 2: height, 3: width},
    },
    verify=True,
    # report=True,
)


Dynamic batch export note

- The issue is in exporting the PyTorch model to ONNX, with dynamic batch size
- Even with dynamic_shapes/dynamic_axes, the exported RT-DETR/PP-DocLayout ONNX model can remain specialized to batch size 1.
- The likely cause is exporter limitation around internal reshape/view operations in transformer-style architectures.
- Typical symptoms:
  - exported ONNX input/output shapes still show batch=1, or
  - ONNXRuntime raises dimension/reshape errors when giving batch > 1.
- Takeaway:
  - dynamic batch sizing is not reliable for this PT -> ONNX export path.
- Safe workaround:
  - run inference one image at a time in a Python loop.
- Alternative workaround:
  - export separate fixed-batch ONNX models such as batch1, batch2, batch4, and use the matching batch size during inference.


#### Verify

In [ ]:
onnx_model = onnx.load(ONNX_SAVE_PATH)
onnx.checker.check_model(onnx_model)

print("ONNX model is valid ✅")

In [ ]:
# ---- Step 1: create SAME (VALID) input ----
image = Image.fromarray(
    np.random.randint(0, 256, (1024, 1024, 3), dtype=np.uint8),
    "RGB"
)

pt_inputs = processor(images=image, return_tensors="pt")
pixel_values = pt_inputs["pixel_values"]

# ---- Step 2: PyTorch output ----
model.eval()
with torch.no_grad():
    outputs = model(pixel_values)
    pt_logits = outputs.logits
    pt_boxes = outputs.pred_boxes

# ---- Step 3: ONNX output ----
ort_session = ort.InferenceSession(ONNX_SAVE_PATH)

onnx_inputs = {
    "pixel_values": pixel_values.numpy()
}

onnx_outputs = ort_session.run(None, onnx_inputs)

onnx_logits = onnx_outputs[0]
onnx_boxes = onnx_outputs[1]

# ---- Step 4: Compare ----
np.testing.assert_allclose(
    pt_logits.cpu().numpy(),
    onnx_logits,
    rtol=1e-2,
    atol=1e-3
)

np.testing.assert_allclose(
    pt_boxes.cpu().numpy(),
    onnx_boxes,
    rtol=1e-2,
    atol=1e-4
)

print("Outputs match ✅")

Note on above check:

Better rule:

if final detections are equivalent, the ONNX export is usable  
exact raw-logit equality is not the right success criterion here  
One more important point:  

this check failure is separate from the dynamic batch issue  
even if single-image ONNX matches “well enough”, dynamic batch can still remain unsupported  
So the fix is:  

relax the tensor comparison, or  
preferably validate at the postprocessed prediction level instead of strict raw output equality.

NOTE:

Check out later how to package model+preprocessor as onnx graph (Confirm if this is a good practise)

and then this approach:

Triton / TensorRT → ONNX + preprocessing fused/optimized

# PRE-PROCESSOR and POST-PROCESSOR

### Reverse Engineering theory

read config → read execution path → extract logic → implement

In [ ]:
print(processor)

In [ ]:
import inspect
print(inspect.getsource(processor.__class__))

processor(images)  
   ↓  
preprocess()  
   ↓  
super().preprocess()  
   ↓  
self._preprocess(...)   ← THIS is what you reimplement


🧠 Why _preprocess exists separately

Separation of concerns:

Function	Role
- preprocess	public API (user calls this)
- _preprocess	actual logic

🧠 Why not call _preprocess directly?

- input validation
- batching
- annotation handling
- format handling

Then delegates to _preprocess.

### preprocessor

Goal: reverse engineer the minimal inference-time preprocessing contract and then reimplement only that logic outside `transformers`.

Keep this mindset throughout:

- first understand the tensor contract
- then trace the execution path
- then discard framework glue
- finally reimplement the smallest correct version

#### Step 1: Inspect and write down the processor configuration values that directly affect preprocessing.

Key point:
Treat the config as the high-level preprocessing contract. It tells you what behavior to look for in the source code.

In [ ]:
print("do_resize:", processor.do_resize)
print("size:", processor.size)
print("do_rescale:", processor.do_rescale)
print("rescale_factor:", processor.rescale_factor)
print("do_normalize:", processor.do_normalize)
print("image_mean:", processor.image_mean)
print("image_std:", processor.image_std)
print("do_pad:", processor.do_pad)

#### Step 2: Inspect the public preprocessing method source code.

Key point:
At this stage, do not try to understand every line. Just identify where the public API delegates the real work.

In [ ]:
import inspect
print(inspect.getsource(processor.preprocess))

#### Step 3: Inspect the parent class method that actually gets called by `super().preprocess(...)`.

Key point:
This is how you discover whether the model-specific processor contains the real logic or just wraps a shared base implementation.

This will show the method resolution order, so we can identify which parent class defines the real `preprocess()` implementation.

In [ ]:
print(processor.__class__.__mro__)

Inspect the `preprocess()` method from `BaseImageProcessor`, because that is the most likely place where the actual call chain starts.

Key point:
This usually reveals the boundary between orchestration code and the actual tensor transformations.

In [ ]:
from transformers.image_processing_utils import BaseImageProcessor
import inspect

print(inspect.getsource(BaseImageProcessor.preprocess))


Now we know the public method still does mostly framework setup and then delegates here:

- validation
- default kwarg handling
- standardization of arguments
- dispatch into the real image-processing path

Key point:
This is framework glue. Useful for understanding control flow, but not what you want to copy into a deployment script.

#### Step 4: Inspect `_preprocess_image_like_inputs()` from `BaseImageProcessor`.

Key point:
This tells you how raw inputs are prepared and where batch-wise image preprocessing begins.

In [ ]:
print(inspect.getsource(BaseImageProcessor._preprocess_image_like_inputs))


#### Step 5: Inspect the model-specific `_preprocess()` implementation on your processor class.

Key point:
This is the most important source block for reverse engineering the preprocessing pipeline.

In [ ]:
print(inspect.getsource(processor.__class__._preprocess))

This is the core preprocessing pipeline.

From here, split the code mentally into two groups:

- inference-time image math you must reproduce
- annotation/training/padding branches you can ignore for now

For your current inference-only use case, focus on:

- image preparation
- resize
- rescale / normalize
- batch stacking

In [ ]:
print(inspect.getsource(processor.__class__.rescale_and_normalize))

In [ ]:
print(inspect.getsource(processor.__class__._fuse_mean_std_and_rescale_factor))


In [ ]:
print(inspect.getsource(processor.__class__.resize))


In [ ]:
print(inspect.getsource(processor.__class__._prepare_image_like_inputs))


In [ ]:
print(inspect.getsource(processor.__class__.process_image))


In [ ]:
print("do_convert_rgb:", getattr(processor, "do_convert_rgb", None))

#### Step 6: Check what happens on a real example after preparation, before resize/rescale.

Key point:
Source code tells you intent. A real tensor inspection confirms the actual contract entering `_preprocess()`.

In [ ]:
prepared = processor._prepare_image_like_inputs(IMG)

print(type(prepared))
print(len(prepared))
print(type(prepared[0]))
print(prepared[0].shape)
print(prepared[0].dtype)
print(prepared[0].min().item(), prepared[0].max().item())


Excellent. Now we have the real input state before _preprocess()

- list of tensors
- per image shape is CHW
- dtype is uint8
- pixel range is [0, 255]

Key takeaway:
By the time `_preprocess()` starts, images are already channel-first tensors. So your custom implementation does not need to copy all of `process_image()` if you choose to work directly from PIL/NumPy in a simpler way.

#### Step 7: Check the actual output after full preprocessing from the official processor.

Key point:
This is the target tensor your custom preprocessing function must match.

In [ ]:
encoded = processor.preprocess(IMG, return_tensors="pt")

print(type(encoded))
print(encoded.keys())
print(encoded["pixel_values"].shape)
print(encoded["pixel_values"].dtype)
print(encoded["pixel_values"].min().item(), encoded["pixel_values"].max().item())

Perfect. You now have the preprocessing target contract:

- output key: pixel_values
- shape: (1, 3, 800, 800)
- dtype: float32
- value range: [0.0, 1.0]

Key takeaway:
If your custom function reproduces this tensor consistently, it is functionally equivalent for inference.

#### Step 8: Create a short written summary, in your own words, of the preprocessing pipeline you have reverse engineered so far.

Key point:
If you can explain the pipeline clearly, you are ready to reimplement it without copying library code line by line.

Write only the minimal inference path, something like:

1. input image is converted to a torch tensor
2. tensor becomes channel-first CHW
3. tensor is uint8 in [0,255]
4. image is resized exactly to 800x800
5. values are converted to float32 and scaled to [0,1]
6. images are stacked into batch dimension
7. final output is pixel_values with shape NCHW

In [ ]:
from typing import Sequence
import numpy as np
from PIL import Image


def preprocess(images: Image.Image | Sequence[Image.Image]) -> np.ndarray:
    """
    Preprocess one or more PIL images into an ONNX-ready batch tensor.

    Reverse-engineered preprocessing contract for this RT-DETR processor:
    - exact resize to 800 x 800
    - no padding
    - pixel values scaled from [0, 255] to [0, 1]
    - no effective normalization beyond rescaling
    - channel order converted from HWC to CHW
    - output batched in NCHW format

    Args:
        images: A single PIL image or a sequence of PIL images.

    Returns:
        A NumPy array of shape (batch_size, 3, 800, 800) with dtype float32.
    """
    if not isinstance(images, (list, tuple)):
        images = [images]

    batch = []
    for image in images:
        # Resize exactly to the processor target size.
        image = image.resize((800, 800))

        # Convert image to NumPy float32 array in HWC layout.
        x = np.array(image, dtype=np.float32)

        # Rescale pixel values from [0, 255] to [0, 1].
        x = x * (1.0 / 255.0)

        # Convert from HWC to CHW layout.
        x = np.transpose(x, (2, 0, 1))

        batch.append(x)

    # Stack images into a batched NCHW tensor.
    batch = np.stack(batch, axis=0).astype(np.float32)

    return batch


Run the custom preprocessor on an image, feed the result to the ONNX model, and compare it with the official processor output.

Important validation idea:

- first verify preprocess parity
- then verify raw ONNX output parity
- only then move on to postprocessing

### post processor

#### Step 1: Inspect the public postprocessing entrypoint source code.

In [ ]:
import inspect
print(inspect.getsource(processor.post_process_object_detection))

#### Step 2: Write down the raw output contract before decoding.

Check and note:

- output tensor names
- logits shape
- pred_boxes shape
- whether pred_boxes are normalized
- whether boxes are in cxcywh or xyxy format

In [ ]:
print(type(outputs))
print(outputs.keys() if hasattr(outputs, "keys") else type(outputs))
print("logits shape:", outputs.logits.shape)
print("pred_boxes shape:", outputs.pred_boxes.shape)

#### Step 3: Trace every helper and math step used inside postprocessing.

From the source, identify:

- activation on logits: sigmoid or softmax
- how top predictions are selected
- how labels are extracted
- how boxes are converted
- how boxes are scaled back to original image size
- how threshold filtering is applied

#### Step 4: Strip away framework glue and keep only inference-time math.

Keep only:

- score computation
- class selection
- box format conversion
- scaling to original image size
- threshold filtering

Ignore:

- argument validation
- optional branches you are not using
- multi-backend support

#### Step 5: Reconstruct the math in plain English before coding.

Write the minimal decoding path in your own words, for example:

1. apply activation to logits
2. rank or filter the predictions
3. get query indices and class ids
4. gather the corresponding boxes
5. convert cxcywh to xyxy if needed
6. scale boxes by original image width and height
7. filter by score threshold
8. return score, label, and box

#### Step 6: Implement a minimal custom postprocess function only after the above is clear.